# 🚂 MARS — Railway Risk & Anomaly Detection
## Model Training Pipeline

This notebook trains **two ML models** for the MARS (Monitoring & Anomaly Recognition System) railway safety platform:

| Model | Purpose | Features |
|---|---|---|
| **Track Risk** | Assess track segment safety from sensor data | 13 features (speed, vibration, temperature, track metadata) |
| **Weather-Track Fusion** | Combined risk from sensors + weather | 20 features (13 track + 7 weather) |

### How to Use
1. **Run all cells** sequentially (Runtime → Run all)
2. Models train on CPU — no GPU needed (~2 minutes total)
3. Trained `.joblib` artifacts will be saved and available for download

---

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q scikit-learn pandas joblib matplotlib seaborn numpy

In [ ]:
import os
import json
import random
import hashlib
import warnings
from datetime import datetime, timedelta, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Create output directories
os.makedirs('artifacts/track-risk', exist_ok=True)
os.makedirs('artifacts/weather-risk', exist_ok=True)
os.makedirs('data', exist_ok=True)

print('✅ All dependencies loaded successfully!')
print(f'📐 scikit-learn version: {__import__("sklearn").__version__}')
print(f'🐼 pandas version: {pd.__version__}')

## 📊 Step 2: Generate Synthetic Training Data

We generate **10,000 rows** of realistic railway sensor and weather data across **50 track segments**.

### Sensor Data Features
- `speed`, `acceleration`, `vibration_vertical`, `vibration_lateral`, `track_temperature`
- Segment metadata: `age_years`, `maintenance_score`, `curvature_degree`, `max_permitted_speed`
- `risk_label`: 0 (normal), 1 (caution), 2 (high risk)

### Weather Data Features
- `rainfall_mm`, `visibility_m`, `temperature_c`, `wind_speed_kmph`
- `hazard_flags`: flood, fog, heat indicators

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Synthetic Data Generator (inline — no external dependencies needed)
# ──────────────────────────────────────────────────────────────────────

SEED = 42
NUM_ROWS = 10000
NUM_SEGMENTS = 50

random.seed(SEED)
np.random.seed(SEED)


def make_segment(index):
    """Create a track segment with realistic metadata."""
    return {
        'segment_id': f'SEG-{index:03d}',
        'region': random.choice(['north', 'south', 'east', 'west', 'central']),
        'age_years': round(random.uniform(2, 65), 2),
        'maintenance_score': round(random.uniform(0.25, 0.98), 3),
        'curvature_degree': round(random.uniform(0, 7), 3),
        'max_permitted_speed': random.choice([60, 80, 100, 110, 130]),
    }


def compute_risk_label(score):
    """Map composite risk score to a 3-class label."""
    if score >= 0.72:
        return 2  # high risk
    elif score >= 0.48:
        return 1  # caution
    return 0      # normal


# Generate segments
segments = [make_segment(i) for i in range(1, NUM_SEGMENTS + 1)]
start_time = datetime(2026, 1, 1, tzinfo=timezone.utc)

# ── Track Sensor Events ──
sensor_rows = []
for i in range(NUM_ROWS):
    seg = random.choice(segments)
    risky = random.random() > seg['maintenance_score'] or seg['age_years'] > 42
    speed = max(0, random.gauss(seg['max_permitted_speed'] * (1.02 if risky else 0.75), 11))
    vertical = max(0.02, random.gauss(0.85 if risky else 0.24, 0.16))
    lateral = max(0.02, random.gauss(0.65 if risky else 0.18, 0.14))
    temp = random.gauss(42 if risky else 32, 6)

    score = (
        min(speed / seg['max_permitted_speed'], 1.4) * 0.22
        + min(vertical, 1.2) * 0.30
        + min(lateral, 1.2) * 0.18
        + (1 - seg['maintenance_score']) * 0.20
        + min(seg['age_years'] / 60, 1) * 0.10
    )

    sensor_rows.append({
        'timestamp': (start_time + timedelta(minutes=i * 5)).isoformat(),
        'segment_id': seg['segment_id'],
        'train_id': f'T-{100 + (i % 80)}',
        'speed': round(speed, 3),
        'acceleration': round(random.gauss(0, 0.9), 3),
        'vibration_vertical': round(vertical, 3),
        'vibration_lateral': round(lateral, 3),
        'track_temperature': round(temp, 3),
        'age_years': seg['age_years'],
        'maintenance_score': seg['maintenance_score'],
        'curvature_degree': seg['curvature_degree'],
        'max_permitted_speed': seg['max_permitted_speed'],
        'risk_label': compute_risk_label(score),
    })

sensor_df = pd.DataFrame(sensor_rows)
sensor_df.to_csv('data/track_sensor_events.csv', index=False)

# ── Weather Events ──
weather_rows = []
for i in range(NUM_ROWS):
    seg = random.choice(segments)
    storm = random.random() < 0.18
    fog = random.random() < 0.12
    heat = random.random() < 0.10
    flags = []
    if storm: flags.append('flood')
    if fog: flags.append('fog')
    if heat: flags.append('heat')

    weather_rows.append({
        'timestamp': (start_time + timedelta(minutes=i * 5)).isoformat(),
        'segment_id': seg['segment_id'],
        'region': seg['region'],
        'rainfall_mm': round(random.uniform(35, 130) if storm else random.uniform(0, 18), 3),
        'visibility_m': round(random.uniform(120, 900) if fog else random.uniform(1200, 12000), 3),
        'temperature_c': round(random.uniform(40, 52) if heat else random.uniform(18, 38), 3),
        'wind_speed_kmph': round(random.uniform(20, 115) if storm else random.uniform(0, 45), 3),
        'hazard_flags': '|'.join(flags),
    })

weather_df = pd.DataFrame(weather_rows)
weather_df.to_csv('data/weather_events.csv', index=False)

print(f'✅ Generated {len(sensor_df):,} sensor events across {sensor_df["segment_id"].nunique()} segments')
print(f'✅ Generated {len(weather_df):,} weather events')
print(f'\n📈 Risk label distribution (sensor data):')
print(sensor_df['risk_label'].value_counts().sort_index().to_string())

In [ ]:
# ── Visualize the generated data ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('MARS Synthetic Dataset Overview', fontsize=16, fontweight='bold', y=1.02)

colors = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
labels = {0: 'Normal', 1: 'Caution', 2: 'High Risk'}

# 1. Risk Label Distribution
ax = axes[0, 0]
counts = sensor_df['risk_label'].value_counts().sort_index()
bars = ax.bar(counts.index, counts.values, color=[colors[i] for i in counts.index], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Risk Label')
ax.set_ylabel('Count')
ax.set_title('Risk Label Distribution')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Normal', 'Caution', 'High Risk'])
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50, f'{count:,}', ha='center', fontweight='bold')

# 2. Speed Distribution by Risk
ax = axes[0, 1]
for risk in [0, 1, 2]:
    subset = sensor_df[sensor_df['risk_label'] == risk]
    ax.hist(subset['speed'], bins=40, alpha=0.6, label=labels[risk], color=colors[risk], edgecolor='white')
ax.set_xlabel('Speed (km/h)')
ax.set_ylabel('Count')
ax.set_title('Speed Distribution by Risk Level')
ax.legend()

# 3. Vibration Scatter
ax = axes[0, 2]
for risk in [0, 1, 2]:
    subset = sensor_df[sensor_df['risk_label'] == risk].sample(min(300, len(sensor_df[sensor_df['risk_label'] == risk])))
    ax.scatter(subset['vibration_vertical'], subset['vibration_lateral'], c=colors[risk],
              alpha=0.4, s=15, label=labels[risk])
ax.set_xlabel('Vibration Vertical (g)')
ax.set_ylabel('Vibration Lateral (g)')
ax.set_title('Vibration Profile by Risk')
ax.legend(markerscale=3)

# 4. Track Temperature by Risk
ax = axes[1, 0]
data_by_risk = [sensor_df[sensor_df['risk_label'] == r]['track_temperature'] for r in [0, 1, 2]]
bp = ax.boxplot(data_by_risk, labels=['Normal', 'Caution', 'High Risk'], patch_artist=True)
for patch, color in zip(bp['boxes'], [colors[0], colors[1], colors[2]]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel('Track Temperature (°C)')
ax.set_title('Temperature by Risk Level')

# 5. Maintenance Score vs Age
ax = axes[1, 1]
scatter = ax.scatter(sensor_df['age_years'], sensor_df['maintenance_score'],
                     c=sensor_df['risk_label'].map(colors), alpha=0.3, s=10)
ax.set_xlabel('Segment Age (years)')
ax.set_ylabel('Maintenance Score')
ax.set_title('Age vs Maintenance by Risk')

# 6. Weather Conditions
ax = axes[1, 2]
weather_summary = pd.DataFrame({
    'Flood': [weather_df['hazard_flags'].str.contains('flood', na=False).sum()],
    'Fog': [weather_df['hazard_flags'].str.contains('fog', na=False).sum()],
    'Heat': [weather_df['hazard_flags'].str.contains('heat', na=False).sum()],
    'Clear': [(weather_df['hazard_flags'] == '').sum()],
}).T
weather_summary.columns = ['Count']
bars = ax.barh(weather_summary.index, weather_summary['Count'],
               color=['#3498db', '#95a5a6', '#e67e22', '#2ecc71'], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Event Count')
ax.set_title('Weather Hazard Distribution')
for bar, count in zip(bars, weather_summary['Count']):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2., f'{int(count):,}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('data/dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Dataset overview saved to data/dataset_overview.png')

---
## 🏗️ Step 3: Define Feature Engineering

The MARS pipeline transforms raw sensor events into model features.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Feature definitions (matching the MARS inference pipeline)
# ──────────────────────────────────────────────────────────────────────

TRACK_FEATURE_ORDER = [
    'speed_mean', 'speed_std', 'speed_max', 'acceleration_abs_mean',
    'vibration_vertical_mean', 'vibration_vertical_std',
    'vibration_lateral_mean', 'vibration_lateral_std',
    'track_temperature_mean', 'segment_age_years', 'maintenance_score',
    'curvature_degree', 'max_permitted_speed',
]

WEATHER_FEATURE_ORDER = [
    *TRACK_FEATURE_ORDER,
    'rainfall_mm', 'visibility_m', 'temperature_c', 'wind_speed_kmph',
    'flood_flag', 'fog_flag', 'heat_flag',
]


def to_track_features(df):
    """Convert raw sensor DataFrame rows to model feature columns.
    
    Note: Since each CSV row is a single event, std features are set to 0.
    At inference time, build_track_features() computes real statistics
    over multi-event windows.
    """
    features = pd.DataFrame(index=df.index)
    features['speed_mean'] = df['speed']
    features['speed_std'] = 0.0
    features['speed_max'] = df['speed']
    features['acceleration_abs_mean'] = df['acceleration'].abs()
    features['vibration_vertical_mean'] = df['vibration_vertical']
    features['vibration_vertical_std'] = 0.0
    features['vibration_lateral_mean'] = df['vibration_lateral']
    features['vibration_lateral_std'] = 0.0
    features['track_temperature_mean'] = df['track_temperature']
    features['segment_age_years'] = df['age_years']
    features['maintenance_score'] = df['maintenance_score']
    features['curvature_degree'] = df['curvature_degree']
    features['max_permitted_speed'] = df['max_permitted_speed']
    return features


def compute_fused_label(row):
    """Combine track risk label with weather severity for a fused target."""
    weather_score = 0
    weather_score += 1 if row['rainfall_mm'] >= 45 else 0
    weather_score += 1 if row['visibility_m'] <= 900 else 0
    weather_score += 1 if row['wind_speed_kmph'] >= 80 else 0
    weather_score += 1 if row['temperature_c'] >= 43 else 0
    return min(2, int(row['risk_label']) + (1 if weather_score >= 2 else 0))


print('✅ Feature engineering functions defined')
print(f'   Track features: {len(TRACK_FEATURE_ORDER)}')
print(f'   Weather features: {len(WEATHER_FEATURE_ORDER)}')

---
## 🚂 Step 4: Train Track Risk Model

**Model**: `GradientBoostingClassifier` wrapped in a `StandardScaler` pipeline  
**Task**: 3-class classification — Normal (0), Caution (1), High Risk (2)  
**Split**: GroupShuffleSplit by segment_id (80/20) to prevent data leakage

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Train Track Risk Model
# ──────────────────────────────────────────────────────────────────────

print('🚂 Training Track Risk Model...')
print('=' * 60)

# Prepare features
track_features = to_track_features(sensor_df)
X_track = track_features[TRACK_FEATURE_ORDER]
y_track = sensor_df['risk_label'].astype(int)
groups_track = sensor_df['segment_id']

# Group-based train/test split (prevents segment leakage)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X_track, y_track, groups_track))

X_train, X_test = X_track.iloc[train_idx], X_track.iloc[test_idx]
y_train, y_test = y_track.iloc[train_idx], y_track.iloc[test_idx]

print(f'📋 Training set: {len(X_train):,} samples')
print(f'📋 Test set:     {len(X_test):,} samples')
print(f'📋 Train segments: {groups_track.iloc[train_idx].nunique()}')
print(f'📋 Test segments:  {groups_track.iloc[test_idx].nunique()}')

# Build and train pipeline
track_model = Pipeline([
    ('scale', StandardScaler()),
    ('classifier', GradientBoostingClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        random_state=42
    )),
])

track_model.fit(X_train, y_train)
track_preds = track_model.predict(X_test)

# Metrics
track_metrics = {
    'accuracy': float(accuracy_score(y_test, track_preds)),
    'precision_macro': float(precision_score(y_test, track_preds, average='macro', zero_division=0)),
    'recall_macro': float(recall_score(y_test, track_preds, average='macro', zero_division=0)),
    'f1_macro': float(f1_score(y_test, track_preds, average='macro', zero_division=0)),
}

print(f'\n🎯 Track Risk Model Results:')
for metric, value in track_metrics.items():
    print(f'   {metric}: {value:.4f}')

print(f'\n📊 Classification Report:')
print(classification_report(y_test, track_preds, target_names=['Normal', 'Caution', 'High Risk']))

In [ ]:
# ── Track Risk Model Visualizations ──
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Track Risk Model — Evaluation', fontsize=16, fontweight='bold', y=1.02)

# 1. Confusion Matrix
ax = axes[0]
cm = confusion_matrix(y_test, track_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal', 'Caution', 'High Risk'],
            yticklabels=['Normal', 'Caution', 'High Risk'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')

# 2. Feature Importance
ax = axes[1]
importances = track_model.named_steps['classifier'].feature_importances_
feat_importance = pd.Series(importances, index=TRACK_FEATURE_ORDER).sort_values(ascending=True)
bars = ax.barh(feat_importance.index, feat_importance.values, color='#3498db', edgecolor='white')
ax.set_xlabel('Feature Importance')
ax.set_title('Feature Importance (Gradient Boosting)')

# 3. Prediction Confidence
ax = axes[2]
track_proba = track_model.predict_proba(X_test)
max_proba = track_proba.max(axis=1)
correct = track_preds == y_test.values
ax.hist(max_proba[correct], bins=30, alpha=0.7, label='Correct', color='#2ecc71', edgecolor='white')
ax.hist(max_proba[~correct], bins=30, alpha=0.7, label='Incorrect', color='#e74c3c', edgecolor='white')
ax.set_xlabel('Max Prediction Probability')
ax.set_ylabel('Count')
ax.set_title('Prediction Confidence')
ax.legend()

plt.tight_layout()
plt.savefig('artifacts/track-risk/evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

# Save model
track_artifact_path = Path('artifacts/track-risk/mars-track-risk.joblib')
joblib.dump(track_model, track_artifact_path)
print(f'\n💾 Track Risk model saved to: {track_artifact_path}')
print(f'   Model size: {track_artifact_path.stat().st_size / 1024:.1f} KB')

---
## 🌦️ Step 5: Train Weather-Track Fusion Model

**Model**: Same pipeline architecture as Track Risk  
**Task**: 3-class classification with fused labels (track risk + weather severity)  
**Features**: 20 features (13 track + 7 weather including hazard flags)

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Train Weather-Track Fusion Model
# ──────────────────────────────────────────────────────────────────────

print('🌦️ Training Weather-Track Fusion Model...')
print('=' * 60)

# Merge sensor and weather data
merged = sensor_df.merge(
    weather_df.drop(columns=['timestamp']),
    on='segment_id',
    how='inner',
    suffixes=('', '_weather'),
)

if len(merged) > 50000:
    print(f'⚠️ Merged dataset is too large ({len(merged):,} rows). Sampling down to 50,000 rows for training...')
    merged = merged.sample(50000, random_state=42).reset_index(drop=True)

print(f'📋 Merged dataset: {len(merged):,} rows')

# Encode weather hazard flags
merged['flood_flag'] = merged['hazard_flags'].fillna('').str.contains('flood').astype(int)
merged['fog_flag'] = merged['hazard_flags'].fillna('').str.contains('fog').astype(int)
merged['heat_flag'] = merged['hazard_flags'].fillna('').str.contains('heat').astype(int)
merged['fused_label'] = merged.apply(compute_fused_label, axis=1)

# Build features
weather_features = to_track_features(merged)
weather_features['rainfall_mm'] = merged['rainfall_mm']
weather_features['visibility_m'] = merged['visibility_m']
weather_features['temperature_c'] = merged['temperature_c']
weather_features['wind_speed_kmph'] = merged['wind_speed_kmph']
weather_features['flood_flag'] = merged['flood_flag']
weather_features['fog_flag'] = merged['fog_flag']
weather_features['heat_flag'] = merged['heat_flag']

X_weather = weather_features[WEATHER_FEATURE_ORDER]
y_weather = merged['fused_label'].astype(int)
groups_weather = merged['segment_id']

# Split
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx_w, test_idx_w = next(splitter.split(X_weather, y_weather, groups_weather))

X_train_w, X_test_w = X_weather.iloc[train_idx_w], X_weather.iloc[test_idx_w]
y_train_w, y_test_w = y_weather.iloc[train_idx_w], y_weather.iloc[test_idx_w]

print(f'📋 Training set: {len(X_train_w):,} samples')
print(f'📋 Test set:     {len(X_test_w):,} samples')

# Build and train pipeline
weather_model = Pipeline([
    ('scale', StandardScaler()),
    ('classifier', GradientBoostingClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        random_state=42
    )),
])

weather_model.fit(X_train_w, y_train_w)
weather_preds = weather_model.predict(X_test_w)

# Metrics
weather_metrics = {
    'accuracy': float(accuracy_score(y_test_w, weather_preds)),
    'precision_macro': float(precision_score(y_test_w, weather_preds, average='macro', zero_division=0)),
    'recall_macro': float(recall_score(y_test_w, weather_preds, average='macro', zero_division=0)),
    'f1_macro': float(f1_score(y_test_w, weather_preds, average='macro', zero_division=0)),
}

print(f'\n🎯 Weather-Track Fusion Model Results:')
for metric, value in weather_metrics.items():
    print(f'   {metric}: {value:.4f}')

print(f'\n📊 Classification Report:')
print(classification_report(y_test_w, weather_preds, target_names=['Normal', 'Caution', 'High Risk']))

In [ ]:
# ── Weather-Track Fusion Model Visualizations ──
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Weather-Track Fusion Model — Evaluation', fontsize=16, fontweight='bold', y=1.02)

# 1. Confusion Matrix
ax = axes[0]
cm_w = confusion_matrix(y_test_w, weather_preds)
sns.heatmap(cm_w, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=['Normal', 'Caution', 'High Risk'],
            yticklabels=['Normal', 'Caution', 'High Risk'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')

# 2. Feature Importance (top 15)
ax = axes[1]
importances_w = weather_model.named_steps['classifier'].feature_importances_
feat_importance_w = pd.Series(importances_w, index=WEATHER_FEATURE_ORDER).sort_values(ascending=True)
colors_fi = ['#e67e22' if f in ['rainfall_mm', 'visibility_m', 'temperature_c', 'wind_speed_kmph',
                                  'flood_flag', 'fog_flag', 'heat_flag'] else '#3498db'
              for f in feat_importance_w.index]
bars = ax.barh(feat_importance_w.index, feat_importance_w.values, color=colors_fi, edgecolor='white')
ax.set_xlabel('Feature Importance')
ax.set_title('Feature Importance (🔵 Track  🟠 Weather)')

# 3. Risk Score Distribution
ax = axes[2]
weather_proba = weather_model.predict_proba(X_test_w)
classes = weather_model.classes_
risk_scores = np.array([sum(float(c) * float(p) for c, p in zip(classes, probs)) / 2.0
                        for probs in weather_proba])
risk_scores = np.clip(risk_scores, 0, 1)

for label, color, name in [(0, '#2ecc71', 'Normal'), (1, '#f39c12', 'Caution'), (2, '#e74c3c', 'High Risk')]:
    mask = y_test_w.values == label
    ax.hist(risk_scores[mask], bins=30, alpha=0.6, label=name, color=color, edgecolor='white')

ax.axvline(x=0.35, color='#f39c12', linestyle='--', linewidth=2, label='Caution threshold')
ax.axvline(x=0.68, color='#e74c3c', linestyle='--', linewidth=2, label='High Risk threshold')
ax.set_xlabel('Risk Score')
ax.set_ylabel('Count')
ax.set_title('Risk Score Distribution')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('artifacts/weather-risk/evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

# Save model
weather_artifact_path = Path('artifacts/weather-risk/mars-weather-risk.joblib')
joblib.dump(weather_model, weather_artifact_path)
print(f'\n💾 Weather-Track model saved to: {weather_artifact_path}')
print(f'   Model size: {weather_artifact_path.stat().st_size / 1024:.1f} KB')

---
## 📋 Step 6: Model Summary & Deployment Cards

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Model Summary Dashboard
# ──────────────────────────────────────────────────────────────────────

print('=' * 70)
print('            🚂 MARS ML Pipeline — Training Summary')
print('=' * 70)

summary = pd.DataFrame({
    'Track Risk': track_metrics,
    'Weather-Track Fusion': weather_metrics,
}).T
summary = summary.round(4)
print('\n📊 Model Performance Comparison:')
print(summary.to_string())

# Side-by-side bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(summary.columns))
width = 0.35
bars1 = ax.bar(x - width/2, summary.iloc[0], width, label='Track Risk', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + width/2, summary.iloc[1], width, label='Weather-Track Fusion', color='#e67e22', edgecolor='white')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['Accuracy', 'Precision\n(macro)', 'Recall\n(macro)', 'F1-Score\n(macro)'])
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.grid(axis='y', alpha=0.3)
for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('artifacts/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Write deployment cards
def write_deployment_card(artifact_dir, model_name, version, artifact_path, metrics, feature_order):
    """Write a deployment governance card."""
    card = {
        'model_name': model_name,
        'model_version': version,
        'artifact_path': str(artifact_path),
        'training_data_snapshot': hashlib.sha256(
            open('data/track_sensor_events.csv', 'rb').read()).hexdigest()[:16],
        'metrics': metrics,
        'feature_order': feature_order,
        'created_at': datetime.now(timezone.utc).isoformat(),
        'training_rows': NUM_ROWS,
        'framework': f'scikit-learn {__import__("sklearn").__version__}',
    }
    card_path = Path(artifact_dir) / 'deployment_card.json'
    card_path.write_text(json.dumps(card, indent=2))
    return card_path

card1 = write_deployment_card('artifacts/track-risk', 'mars-track-risk', '0.1.0',
                              track_artifact_path, track_metrics, TRACK_FEATURE_ORDER)
card2 = write_deployment_card('artifacts/weather-risk', 'mars-weather-risk', '0.1.0',
                              weather_artifact_path, weather_metrics, WEATHER_FEATURE_ORDER)

print(f'\n📝 Deployment cards written:')
print(f'   {card1}')
print(f'   {card2}')

---
## ⬇️ Step 7: Download Model Artifacts

Run this cell to download the trained models and deployment cards.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Download artifacts (works in Google Colab)
# ──────────────────────────────────────────────────────────────────────

print('📦 Artifacts ready for download:')
print(f'   ├── artifacts/track-risk/mars-track-risk.joblib ({track_artifact_path.stat().st_size / 1024:.1f} KB)')
print(f'   ├── artifacts/track-risk/deployment_card.json')
print(f'   ├── artifacts/track-risk/evaluation.png')
print(f'   ├── artifacts/weather-risk/mars-weather-risk.joblib ({weather_artifact_path.stat().st_size / 1024:.1f} KB)')
print(f'   ├── artifacts/weather-risk/deployment_card.json')
print(f'   ├── artifacts/weather-risk/evaluation.png')
print(f'   └── artifacts/model_comparison.png')

try:
    from google.colab import files
    print('\n🔽 Downloading model artifacts...')
    files.download('artifacts/track-risk/mars-track-risk.joblib')
    files.download('artifacts/weather-risk/mars-weather-risk.joblib')
    files.download('artifacts/track-risk/deployment_card.json')
    files.download('artifacts/weather-risk/deployment_card.json')
    print('✅ Download complete!')
except ImportError:
    print('\n💡 Not running in Colab — artifacts saved to local `artifacts/` directory.')
    print('   You can find them at the paths listed above.')

---
## 🧪 Step 8: Quick Inference Test

Verify both models work with sample input.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Quick inference test
# ──────────────────────────────────────────────────────────────────────

def predict_risk_score(model, features_dict, feature_order):
    """Compute a calibrated risk score in [0, 1] from model probabilities."""
    row = pd.DataFrame([{name: float(features_dict[name]) for name in feature_order}])
    probabilities = model.predict_proba(row)[0]
    classes = list(model.classes_)
    weighted = sum(float(cls) * float(prob) for cls, prob in zip(classes, probabilities))
    return max(0.0, min(1.0, weighted / 2.0))


def classify_severity(score):
    """Map risk score to severity level."""
    if score >= 0.68:
        return '🔴 HIGH RISK', 2, 'Restrict Speed'
    elif score >= 0.35:
        return '🟡 CAUTION', 1, 'Caution Advisory'
    return '🟢 NORMAL', 0, 'Normal Operations'


# Test Case 1: Normal conditions
normal_input = {
    'speed_mean': 75, 'speed_std': 0, 'speed_max': 75,
    'acceleration_abs_mean': 0.2,
    'vibration_vertical_mean': 0.2, 'vibration_vertical_std': 0,
    'vibration_lateral_mean': 0.15, 'vibration_lateral_std': 0,
    'track_temperature_mean': 30,
    'segment_age_years': 10, 'maintenance_score': 0.85,
    'curvature_degree': 2, 'max_permitted_speed': 110,
}

# Test Case 2: Risky conditions
risky_input = {
    'speed_mean': 115, 'speed_std': 0, 'speed_max': 115,
    'acceleration_abs_mean': 1.2,
    'vibration_vertical_mean': 0.85, 'vibration_vertical_std': 0,
    'vibration_lateral_mean': 0.7, 'vibration_lateral_std': 0,
    'track_temperature_mean': 48,
    'segment_age_years': 45, 'maintenance_score': 0.3,
    'curvature_degree': 5, 'max_permitted_speed': 100,
}

print('🧪 Inference Tests')
print('=' * 60)

for label, test_input in [('Normal conditions', normal_input), ('Risky conditions', risky_input)]:
    score = predict_risk_score(track_model, test_input, TRACK_FEATURE_ORDER)
    severity, risk_class, action = classify_severity(score)
    print(f'\n📍 {label}:')
    print(f'   Risk Score:  {score:.4f}')
    print(f'   Severity:    {severity}')
    print(f'   Risk Class:  {risk_class}')
    print(f'   Action:      {action}')

# Weather test
weather_risky = {**risky_input,
    'rainfall_mm': 80, 'visibility_m': 400, 'temperature_c': 46,
    'wind_speed_kmph': 90, 'flood_flag': 1, 'fog_flag': 1, 'heat_flag': 1,
}
w_score = predict_risk_score(weather_model, weather_risky, WEATHER_FEATURE_ORDER)
w_severity, w_class, w_action = classify_severity(w_score)
print(f'\n📍 Risky + Severe Weather:')
print(f'   Risk Score:  {w_score:.4f}')
print(f'   Severity:    {w_severity}')
print(f'   Risk Class:  {w_class}')
print(f'   Action:      {w_action}')

print('\n✅ All inference tests passed!')
print('\n🎉 Training pipeline complete! Models are ready for deployment.')